In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Reproducibility
np.random.seed(42)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Reproducible random generator
rng = np.random.default_rng(42)

# Historical period: 104 weeks
weeks = pd.date_range(
    start="2024-11-04",
    periods=104,
    freq="W-MON"
)

# Courses and learner segments
courses = [
    "Data Science",
    "Python Programming",
    "Machine Learning",
    "Cloud Computing",
    "Business Analytics",
    "Cyber Security",
    "Digital Marketing",
    "Web Development"
]

segments = [
    "College Students",
    "Working Professionals",
    "Distance Learners"
]

# Alternate course names across academic terms
course_aliases = {
    "Data Science": ["Data Science", "Applied Data Science"],
    "Python Programming": ["Python Programming", "Python for Analytics"],
    "Machine Learning": ["Machine Learning", "Applied Machine Learning"],
    "Cloud Computing": ["Cloud Computing", "Cloud Technologies"],
    "Business Analytics": ["Business Analytics", "Business Intelligence"],
    "Cyber Security": ["Cyber Security", "Information Security"],
    "Digital Marketing": ["Digital Marketing", "Online Marketing"],
    "Web Development": ["Web Development", "Full Stack Development"]
}

# Base weekly demand for each course
course_base = np.array([75, 85, 65, 55, 70, 50, 60, 80])

# Relative demand across learner segments
segment_factor = np.array([1.0, 0.85, 0.75])

# Generate weekly demand with trend and seasonality
raw_demand = np.zeros((len(weeks), len(courses), len(segments)))

for w, week in enumerate(weeks):

    annual_seasonality = (
        1 + 0.15 * np.sin(2 * np.pi * (w % 52) / 52)
    )

    long_term_trend = 1 + 0.0015 * w

    admission_peak = 1.35 if week.month in [1, 7, 10] else 1.0

    festival_effect = 1.15 if week.month in [3, 10, 11] else 1.0

    for c in range(len(courses)):
        for s in range(len(segments)):

            noise = rng.normal(1.0, 0.12)

            raw_demand[w, c, s] = (
                course_base[c]
                * segment_factor[s]
                * annual_seasonality
                * long_term_trend
                * admission_peak
                * festival_effect
                * max(noise, 0.5)
            )

# Scale demand to exactly 400,000 enrolment records
scaled_demand = raw_demand * (400000 / raw_demand.sum())

weekly_counts = np.floor(scaled_demand).astype(int)

remaining = 400000 - weekly_counts.sum()

fractional_order = np.argsort(
    (scaled_demand - weekly_counts).ravel()
)[::-1]

weekly_counts.ravel()[fractional_order[:remaining]] += 1

# Course fee reference values
course_fees = {
    "Data Science": 18000,
    "Python Programming": 12000,
    "Machine Learning": 20000,
    "Cloud Computing": 16000,
    "Business Analytics": 15000,
    "Cyber Security": 19000,
    "Digital Marketing": 10000,
    "Web Development": 14000
}

# Generate learner-level records
records = []
learner_counter = 1

for w, week in enumerate(weeks):

    year = week.year
    term_number = 1 if week.month <= 6 else 2
    term = f"{year}-T{term_number}"

    promotion_active = rng.random() < 0.12

    for c, course in enumerate(courses):

        # Alternate displayed course names between terms
        alias_index = (year + term_number) % 2
        course_name = course_aliases[course][alias_index]

        for s, segment in enumerate(segments):

            n = weekly_counts[w, c, s]

            if n == 0:
                continue

            # Random enrolment dates within the week
            enrolment_dates = (
                week + pd.to_timedelta(
                    rng.integers(0, 7, n), unit="D"
                )
            )

            # Session activity
            recorded_sessions = rng.integers(0, 9, n)

            attended_sessions = rng.integers(
                0, recorded_sessions + 1
            )

            attendance_pct = np.divide(
                attended_sessions,
                recorded_sessions,
                out=np.full(n, np.nan),
                where=recorded_sessions > 0
            ) * 100

            # Introduce missing attendance for recorded sessions
            missing_mask = rng.random(n) < 0.10
            attendance_pct[missing_mask] = np.nan

            # Mixed score scales: 50 and 100
            assignment_max = rng.choice([50, 100], n)
            assessment_max = rng.choice([50, 100], n)

            attendance_for_score = np.nan_to_num(
                attendance_pct, nan=65
            )

            assignment_fraction = np.clip(
                0.40 + 0.0035 * attendance_for_score
                + rng.normal(0, 0.12, n),
                0.05, 1.0
            )

            assessment_fraction = np.clip(
                0.35 + 0.004 * attendance_for_score
                + rng.normal(0, 0.13, n),
                0.05, 1.0
            )

            assignment_score = np.round(
                assignment_max * assignment_fraction
            )

            assessment_score = np.round(
                assessment_max * assessment_fraction
            )

            # Session dates after enrolment
            session_dates = (
                enrolment_dates
                + pd.to_timedelta(
                    rng.integers(0, 31, n), unit="D"
                )
            )

            # Course fee and promotional pricing
            original_fee = course_fees[course]

            course_fee = (
                original_fee * 0.90
                if promotion_active
                else original_fee
            )

            # Festival indicators
            holi_period = week.month == 3
            diwali_period = week.month in [10, 11]

            batch = pd.DataFrame({
                "learner_id": [
                    f"L{i:07d}"
                    for i in range(
                        learner_counter,
                        learner_counter + n
                    )
                ],
                "enrolment_date": enrolment_dates,
                "session_date": session_dates,
                "enrolment_week": week,
                "course_id": f"C{c+1:02d}",
                "course_name": course_name,
                "course_segment": course,
                "learner_segment": segment,
                "academic_term": term,
                "recorded_sessions": recorded_sessions,
                "attended_sessions": attended_sessions,
                "attendance_pct": attendance_pct,
                "assignment_score": assignment_score,
                "assignment_max": assignment_max,
                "assessment_score": assessment_score,
                "assessment_max": assessment_max,
                "course_fee": course_fee,
                "promotion_active": promotion_active,
                "holi_period": holi_period,
                "diwali_period": diwali_period,
                "new_enrolment": 1
            })

            records.append(batch)
            learner_counter += n

# Combine all records
df = pd.concat(records, ignore_index=True)

# Sort chronologically
df = df.sort_values(
    "enrolment_date"
).reset_index(drop=True)

print("Dataset generated successfully!")
print("Total records:", len(df))
print("Total columns:", len(df.columns))


Dataset generated successfully!
Total records: 400000
Total columns: 21


In [3]:

print("Dataset shape:", df.shape)

print("\nFirst 5 records:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nUnique courses:")
print(df["course_segment"].nunique())

print("\nUnique learner segments:")
print(df["learner_segment"].unique())

print("\nScore scales:")
print("Assignment:", sorted(df["assignment_max"].unique()))
print("Assessment:", sorted(df["assessment_max"].unique()))

print("\nDate range:")
print(df["enrolment_date"].min())
print(df["enrolment_date"].max())

assert len(df) == 400000
assert df["learner_id"].nunique() == 400000

print("\nAll basic validation checks passed!")


Dataset shape: (400000, 21)

First 5 records:


,learner_id,enrolment_date,session_date,enrolment_week,course_id,course_name,course_segment,learner_segment,academic_term,recorded_sessions,...,attendance_pct,assignment_score,assignment_max,assessment_score,assessment_max,course_fee,promotion_active,holi_period,diwali_period,new_enrolment
0,L0003224,2024-11-04,2024-11-24,2024-11-04,C08,Web Development,Web Development,College Students,2024-T2,3,...,100.000000,73.0,100,43.0,50,14000.0,False,False,True,1
1,L0001035,2024-11-04,2024-11-28,2024-11-04,C02,Python Programming,Python Programming,Distance Learners,2024-T2,7,...,85.714286,74.0,100,69.0,100,12000.0,False,False,True,1
2,L0001038,2024-11-04,2024-12-02,2024-11-04,C02,Python Programming,Python Programming,Distance Learners,2024-T2,3,...,66.666667,69.0,100,69.0,100,12000.0,False,False,True,1
3,L0001043,2024-11-04,2024-12-04,2024-11-04,C02,Python Programming,Python Programming,Distance Learners,2024-T2,2,...,0.000000,13.0,50,18.0,50,12000.0,False,False,True,1
4,L0001046,2024-11-04,2024-11-15,2024-11-04,C03,Machine Learning,Machine Learning,College Students,2024-T2,1,...,100.000000,100.0,100,73.0,100,20000.0,False,False,True,1



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400000 entries, 0 to 399999
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   learner_id         400000 non-null  object        
 1   enrolment_date     400000 non-null  datetime64[ns]
 2   session_date       400000 non-null  datetime64[ns]
 3   enrolment_week     400000 non-null  datetime64[ns]
 4   course_id          400000 non-null  object        
 5   course_name        400000 non-null  object        
 6   course_segment     400000 non-null  object        
 7   learner_segment    400000 non-null  object        
 8   academic_term      400000 non-null  object        
 9   recorded_sessions  400000 non-null  int64         
 10  attended_sessions  400000 non-null  int64         
 11  attendance_pct     320010 non-null  float64       
 12  assignment_score   400000 non-null  float64       
 13  assignment_max     400

,0
learner_id,0
enrolment_date,0
session_date,0
enrolment_week,0
course_id,0
course_name,0
course_segment,0
learner_segment,0
academic_term,0
recorded_sessions,0



Unique courses:
8

Unique learner segments:
['College Students' 'Distance Learners' 'Working Professionals']

Score scales:
Assignment: [np.int64(50), np.int64(100)]
Assessment: [np.int64(50), np.int64(100)]

Date range:
2024-11-04 00:00:00
2026-11-01 00:00:00

All basic validation checks passed!


In [4]:

df.to_csv("edtech_synthetic_400k.csv", index=False)

print("Dataset saved successfully!")


Dataset saved successfully!
